In [1]:
import sys
!{sys.executable} -m pip install hdfs

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta import configure_spark_with_delta_pip

# 1. Configuração da Sessão
builder = SparkSession.builder \
    .appName("Gold_Dim_Calendario") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()
spark = configure_spark_with_delta_pip(builder).getOrCreate()

gold_calendario_path = "hdfs://hdfs-nn:9000/demo/gold/dim_calendario"

ano_inicio = 1900
ano_fim = 2025

print(f"A criar a Tabela Dimensão Calendário do ano {ano_inicio} até ao ano {ano_fim}\n")

# 2. Geração dos dados
# O 'id' do range começa em 1900, mas vamos transformá-lo num ID inteiro sequencial
dim_calendario = spark.range(ano_inicio, ano_fim + 1) \
    .withColumn("ano", col("id").cast("int")) \
    .withColumn("id_calendario", (col("id") - (ano_inicio - 1)).cast("int")) \
    .select("id_calendario", "ano")

# 3. Gestão do Catálogo (Adicionado DROP TABLE para limpeza)
spark.sql("CREATE DATABASE IF NOT EXISTS Gold_db")
spark.sql("DROP TABLE IF EXISTS Gold_db.dim_calendario")

print(f"-> A Gravar Dimensão Calendário em {gold_calendario_path}")

# 4. Gravação em Delta
dim_calendario.repartition(1).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_calendario_path)

# Registo no Hive
spark.sql(f"CREATE TABLE Gold_db.dim_calendario USING DELTA LOCATION '{gold_calendario_path}'")

print(f"\n-> Tabela Dimensão Calendário (ID Inteiro)")
dim_calendario.orderBy(col("ano").desc()).show(10)

spark.stop()

A criar a Tabela Dimensão Calendário do ano 1900 até ao ano 2025

-> A Gravar Dimensão Calendário em hdfs://hdfs-nn:9000/demo/gold/dim_calendario
